# Retinal age prediction from fundus images - full experiment run

TAU Deep Learning, final project. Shahaf Gino & Adi Dror.

This notebook runs every experiment in the project on a Kaggle GPU session.
It is the notebook the reported results came from, kept in the order the
work was actually done.

**Session settings (right-hand panel):** Accelerator `GPU T4 x2`, Internet off.

**Data sources to attach:**
1. the output of `kaggle_01_prepare_data` (the `splits/` CSVs and the `images_224` cache)
2. the `Retina-code` dataset (this repo's `src/` folder, and any earlier `metrics/` and `checkpoints/`)

The full matrix is 12 CNN configurations x 3 seeds plus two baselines, about
2.5 hours of GPU time. `train_cnn` skips any run whose metrics file already
exists, so the notebook is safe to re-run and safe to resume after a
disconnect.

## 1. Environment and code

In [1]:
import torch, os, sys
print('CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')
print('CPUs:', os.cpu_count())
!ls /kaggle/input


CUDA: True Tesla T4
CPUs: 4
datasets  notebooks


In [3]:
import shutil
from pathlib import Path

# find the attached folder that contains src/
code_src = next(p for p in Path('/kaggle/input').rglob('src') if (p/'train.py').exists())
print('found code at:', code_src)

work = Path('/kaggle/working/project')
work.mkdir(parents=True, exist_ok=True)
shutil.copytree(code_src, work/'src', dirs_exist_ok=True)
sys.path.insert(0, str(work))

for m in [k for k in list(sys.modules) if k == 'src' or k.startswith('src.')]:
    del sys.modules[m]
print('ok')


found code at: /kaggle/input/datasets/shahafgino/retina-code/src
ok


In [7]:
import src.dataset as ds, src.train as tr, inspect
print("depth-independent :", '["*"] * n' in inspect.getsource(ds.find_prepared_dataset))
print("baseline uses cache:", "if not return_features" not in inspect.getsource(ds.get_dataloaders))
print("huber delta        :", tr.HUBER_DELTA, "(must be 5.0, not 1.0)")

root = ds.download_dataset()
imgs = ds.resolve_images_dir(root, 224)
print("images dir         :", imgs)
print("jpgs               :", len(list(imgs.glob('*.jpg'))), "(expect 9857)")

depth-independent : True
baseline uses cache: True
huber delta        : 5.0 (must be 5.0, not 1.0)
images dir         : /kaggle/input/notebooks/shahafgino/notebookd4fa558927/retina224/images_224
jpgs               : 9857 (expect 9857)


In [6]:
# render progress bars as notebook widgets instead of thousands of text lines
from tqdm.auto import tqdm as _auto_tqdm
import src.train as _tr
_tr.tqdm = _auto_tqdm
print("progress bars -> notebook widgets")

progress bars -> notebook widgets


Earlier sessions' results are attached read-only; copy them somewhere writable.

In [ ]:
# ---- Seed the working results dir from the dataset ----
# Earlier sessions' metrics (and checkpoints, when the dataset ships them)
# live in the read-only input dir. Copy whatever is actually there, so this
# works whether or not the current dataset version includes checkpoints.
import shutil
from pathlib import Path
from src.utils import CHECKPOINTS_DIR, METRICS_DIR

src_root = next(p for p in Path("/kaggle/input").rglob("src")
                if (p / "train.py").exists()).parent
print("dataset root:", src_root)

for sub, dest in [("checkpoints", CHECKPOINTS_DIR), ("metrics", METRICS_DIR)]:
    folder = src_root / sub
    if not folder.is_dir():
        print(f"  no {sub}/ in this dataset version - skipping")
        continue
    n = 0
    for f in folder.iterdir():
        shutil.copy2(f, dest / f.name); n += 1
    print(f"  copied {n} files -> {dest}")

## 2. Blocking checks

Patient overlap must be zero across all three splits, or nothing below means
anything. The mean predictor is the bar every model has to clear.

In [13]:
from src.train import check_patient_leakage, run_mean_predictor
check_patient_leakage()
run_mean_predictor()


  train n val   : 0
  train n test  : 0
  val n test    : 0
  patients per split: {'train': 3775, 'val': 809, 'test': 809}
  OK - no patient appears in more than one split.


{'MAE': 14.31643021848581,
 'RMSE': 18.02531522712326,
 'R2': -4.6627621072214964e-05,
 'train_mean_age': 57.423355549116195}

## 3. Non-deep baseline

Ridge regression on hand-crafted features. Deterministic, so one seed is enough.

In [14]:
from src.train import train_linear_baseline
train_linear_baseline()


ridge_linear: train features:   0%|          | 0/216 [00:00<?, ?it/s]

ridge_linear: test features:   0%|          | 0/46 [00:00<?, ?it/s]

[ridge_linear] MAE=9.55  RMSE=12.64  R2=0.508


{'MAE': 9.55331039428711,
 'RMSE': 12.644558401847586,
 'R2': 0.5078904032707214,
 'seed': 0}

## 4. Isolating the regularization

Each configuration differs from `cnn_plain` in exactly one factor. Our first
attempt changed six things at once and the comparison meant nothing.

| config | one factor vs cnn_plain |
|---|---|
| `cnn_dropout` | dropout |
| `cnn_wd` | weight decay |
| `cnn_es` | early stopping (best-val weights instead of final) |
| `cnn_all` | all three together |

In [ ]:
from src.train import run_experiment

# Each differs from cnn_plain in exactly one factor.
for name in ["cnn_plain", "cnn_dropout", "cnn_wd", "cnn_es", "cnn_all"]:
    run_experiment(name, seeds=[0, 1, 2])

In [16]:
from src.train import aggregate_results
aggregate_results()


,n_seeds,MAE,MAE_std,RMSE,R2,params
config,,,,,,
cnn_wd,3,7.655,0.157,10.154,0.6826,360321.0
cnn_es,3,7.970,0.305,10.587,0.6546,360321.0
cnn_plain,3,8.129,0.490,10.664,0.6495,360321.0
cnn_dropout,3,8.600,0.606,11.187,0.6141,360321.0
cnn_all,3,8.672,0.085,11.457,0.5959,360321.0
ridge_linear,1,9.553,NaN,12.645,0.5079,NaN
mean_predictor,1,14.316,NaN,18.025,-0.0000,NaN


## 5. Architecture ablations

These are anchored to `cnn_all`, each varying one factor. `arch_7x7` has its
channel width matched by bisection so its parameter count matches the anchor
to within 0.25%; `arch_valid` is parameter-identical because the adaptive
pool makes the padding mode irrelevant to the head size.

In [12]:
from src.train import run_experiment

for name in ["cnn_aug", "arch_7x7", "arch_valid", "arch_drop_early"]:
    run_experiment(name, seeds=[0, 1, 2])


cnn_aug   (vs cnn_all: augmentation)
[cnn_aug_seed0] 360,321 params | loss=mse | wd=0.0001 | select=best-val


  train batches:   0%|          | 0/216 [00:00<?, ?it/s]

  epoch 1/20  train=308.74  val=184.14  (19s)


  train batches:   0%|          | 0/216 [00:00<?, ?it/s]

  epoch 2/20  train=199.58  val=183.01  (12s)


...

  epoch 18/20  train=157.08  val=151.53  (11s)


  train batches:   0%|          | 0/216 [00:00<?, ?it/s]

  epoch 19/20  train=157.86  val=151.33  (11s)


  train batches:   0%|          | 0/216 [00:00<?, ?it/s]

  epoch 20/20  train=158.31  val=163.71  (11s)
  early stop at epoch 20 (best val 151.22)
[arch_drop_early_seed2] MAE=9.17  RMSE=12.19  R2=0.543


## 6. Loss function

Same architecture and schedule as `cnn_all`; only the loss changes. Huber uses
`delta=5.0` years - at PyTorch's default of 1.0 it would sit in its linear
regime for every sample and be numerically identical to L1.

In [9]:
# ---- Session 3, part A: the loss-function ablation ----
# Single varying factor vs the cnn_all anchor: the loss only. Same
# architecture, same epochs, same weight decay, same checkpoint rule.
from src.train import run_experiment

for name in ["loss_l1", "loss_huber"]:
    run_experiment(name)


loss_l1   (vs cnn_all: L1 loss)
[loss_l1_seed0] 360,321 params | loss=l1 | wd=0.0001 | select=best-val


  train batches:   0%|          | 0/216 [00:00<?, ?it/s]

  epoch 1/20  train=13.03  val=10.25  (25s)


  train batches:   0%|          | 0/216 [00:00<?, ?it/s]

  epoch 2/20  train=10.86  val=9.33  (11s)


...

  epoch 18/20  train=33.35  val=45.77  (11s)


  train batches:   0%|          | 0/216 [00:00<?, ?it/s]

  epoch 19/20  train=32.89  val=39.51  (11s)


  train batches:   0%|          | 0/216 [00:00<?, ?it/s]

  epoch 20/20  train=33.45  val=28.80  (11s)
[loss_huber_seed2] MAE=7.54  RMSE=10.08  R2=0.687


## 7. Does a larger epoch budget help?

A follow-up question raised by the results above, not part of the original plan.

In [ ]:
# cnn_aug hit the 20-epoch ceiling in all three seeds without early
# stopping ever firing, so its result looked like a lower bound. This run
# changes ONE thing - the epoch budget, 20 -> 40 - to find out whether the
# ceiling was really the limiting factor.
from src.train import run_experiment
run_experiment("cnn_aug_long")

## 8. Post-hoc analyses

No training here - each of these loads a saved checkpoint and runs inference.

In [11]:
# ---- Session 3, part B: post-hoc analyses ----
# No training here. Everything below loads a saved checkpoint and runs
# inference, so it is cheap and safe to re-run.
from src.analysis import run_age_analysis
from src.calibration import run_calibration_analysis
from src.saliency import run_saliency_analysis
from src.biomarker import run_biomarker_analysis

# 1. Where does the error live? Per-decade MAE vs signed bias.
run_age_analysis("cnn_all", seed=0)

# 2. Can a linear correction fitted on validation fix the compression?
run_calibration_analysis("cnn_all", seed=0)

# 3. Where does the model look? Compares the two receptive-field configs,
#    which is what our 7x7-vs-3x3 ablation assumes differs.
run_saliency_analysis(["cnn_all", "arch_7x7"], seed=0)

# 4. Is the corrected age gap related to the clinical labels?
run_biomarker_analysis("cnn_all", seed=0)

[cnn_all] running inference on the test set (no training)

overall MAE=8.74  overall bias=+1.29
age_group   n       MAE       bias       std  reliable
      0-9   8 36.821579  36.821579  9.993157     False
    10-19  57 14.695210  14.450953 12.355467      True
    20-29  71 14.098946  13.426780 11.843280      True
    30-39 123 11.618666  10.817962  8.678925      True
    40-49 153  8.441222   7.548068  8.112410      True
    50-59 289  6.044599   3.454389  7.059325      True
    60-69 362  6.221416  -0.457680  8.059205      True
    70-79 288  8.193388  -6.759600  8.238678      True
    80-89 103 13.846208 -13.736232  8.426237      True
    90-99   8 17.072279 -17.072279  6.949454     False

saved cnn_all_error_by_age_group.csv and 3 figures to results/
[cnn_all] predicting on validation (to fit the corrections)
[cnn_all] predicting on test (to evaluate them)

measured compression on validation: pred = 0.569 * true + ...
  the model spans only 57% of the real age range
A (MSE-optimal)

(      image_id  patient_id  patient_age     age_group  age_group_numeric  \
 0     img00003           2         18.0   young_adult                  1   
 1     img00004           2         18.0   young_adult                  1   
 2     img00009           5         23.0   young_adult                  1   
 3     img00010           5         23.0   young_adult                  1   
 4     img00051          26         13.0     pediatric                  0   
 ...        ...         ...          ...           ...                ...   
 1457  img16262        8522         33.0  middle_age_1                  2   
 1458  img16263        8523         19.0   young_adult                  1   
 1459  img16264        8523         19.0   young_adult                  1   
 1460  img16265        8524         20.0   young_adult                  1   
 1461  img16266        8524         20.0   young_adult                  1   
 
      age_group_broad  age_group_broad_numeric  patient_sex  exam_eye  \
 

## 9. Summary and save

In [10]:
# ---- Summary of every configuration run so far ----
import json, glob, os, statistics as st

runs = {}
for f in glob.glob("/kaggle/working/project/results/metrics/*.json"):
    name = os.path.basename(f)[:-5]
    base = name.rsplit("_seed", 1)[0] if "_seed" in name else name
    runs.setdefault(base, []).append(json.load(open(f))["MAE"])

print(f"{'config':20} {'MAE':>8} {'sd':>6} {'seeds':>5}")
for k, v in sorted(runs.items(), key=lambda kv: st.mean(kv[1])):
    sd = st.pstdev(v) if len(v) > 1 else 0.0
    print(f"{k:20} {st.mean(v):8.3f} {sd:6.3f} {len(v):5d}")

config                    MAE     sd seeds
cnn_aug                 7.638  0.108     3
cnn_wd                  7.655  0.128     3
loss_huber              7.722  0.220     3
arch_valid              7.888  0.440     3
cnn_es                  7.970  0.249     3
arch_7x7                8.018  0.065     3
loss_l1                 8.109  0.116     3
cnn_plain               8.129  0.400     3
cnn_dropout             8.600  0.495     3
cnn_all                 8.672  0.070     3
arch_drop_early         9.051  0.084     3
ridge_linear            9.553  0.000     1
mean_predictor         14.316  0.000     1


In [ ]:
# ---- Pack the results into one downloadable zip ----
import shutil, os
from pathlib import Path
from src.utils import RESULTS_DIR, METRICS_DIR, CHECKPOINTS_DIR, FIGURES_DIR

SESSION_NAME = "results"

print(f"metrics json : {len(list(METRICS_DIR.glob('*.json')))}")
print(f"csv files    : {len(list(METRICS_DIR.glob('*.csv')))}")
print(f"checkpoints  : {len(list(CHECKPOINTS_DIR.glob('*.pt')))}")
print(f"figures      : {len(list(FIGURES_DIR.glob('*.png')))}")

out = shutil.make_archive(str(Path("/kaggle/working") / SESSION_NAME),
                          "zip", root_dir=str(RESULTS_DIR))
print(f"\ncreated {out}  ({os.path.getsize(out)/1e6:.1f} MB)")